In [1]:
def parse_solution_file(file_path):
    import re
    from collections import defaultdict

    routes = {'x': defaultdict(list), 'y': defaultdict(list), 'z': defaultdict(list)}
    ids_seen = {'x': set(), 'y': set(), 'z': set()}

    def extract_suffix(ent_id: str) -> int:
        match = re.search(r'(\d+)$', ent_id)
        return int(match.group(1)) if match else None

    def valid_node(node):
        try:
            return int(node)
        except ValueError:
            return None

    with open(file_path, 'r') as file:
        for line in file:
            if ' 1' not in line or not line.startswith(('x[', 'y[', 'z[')):
                continue

            var_type = line[0]
            key = line.split('[')[1].split(']')[0]
            parts = key.split(',')

            if len(parts) != 3:
                continue

            ent, f, t = parts
            ent_id = extract_suffix(ent)
            if ent_id is None:
                continue

            # Nur gültige int-Knoten übernehmen
            from_node = valid_node(f)
            to_node = valid_node(t)

            if from_node is not None and to_node is not None:
                routes[var_type][ent_id].append((from_node, to_node))
            elif from_node is None and to_node is not None:
                routes[var_type][ent_id].append(('start', to_node))
            elif from_node is not None and to_node is None:
                routes[var_type][ent_id].append((from_node, 'end'))

            ids_seen[var_type].add(ent_id)

    def build_path(transitions):
        adj = defaultdict(list)
        incoming = defaultdict(int)

        for f, t in transitions:
            if isinstance(f, int) and isinstance(t, int):
                adj[f].append(t)
                incoming[t] += 1

        all_nodes = set(n for pair in transitions for n in pair if isinstance(n, int))
        start_nodes = [n for n in all_nodes if incoming[n] == 0]

        full_path = []
        visited = set()
        for start in start_nodes:
            current = start
            while current in adj and adj[current]:
                next_node = adj[current].pop(0)
                if (current, next_node) not in visited:
                    full_path.append((current, next_node))
                    visited.add((current, next_node))
                    current = next_node
                else:
                    break

        path = [full_path[0][0]] + [t for _, t in full_path] if full_path else []

        # Aufträge aus 'start' und 'end' Übergängen hinzufügen
        for f, t in transitions:
            if f == 'start' and isinstance(t, int) and t not in path:
                path.insert(0, t)
            elif t == 'end' and isinstance(f, int) and f not in path:
                path.append(f)

        return path

    final_routes = {}
    for k in ['x', 'y', 'z']:
        max_id = max(ids_seen[k]) if ids_seen[k] else -1
        filled = {}
        for i in range(max_id + 1):
            transitions = routes[k].get(i, [])
            filled[i] = build_path(transitions) if transitions else []
        final_routes[k] = dict(sorted(filled.items()))

    return final_routes


# Beispiel
solution_path = "solution_a5_o96_m10_an10_ar10_reduced.sol"
routes = parse_solution_file(solution_path)

print("Maschinenrouten (x):")
print(routes['x'])

print("\nArbeiterrouten (y):")
print(routes['y'])

print("\nAnbaugeräte-Routen (z):")
print(routes['z'])



Maschinenrouten (x):
{0: [], 1: [0, 2, 10, 14, 15, 17, 18, 21, 22, 24, 34, 38], 2: [46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79], 3: [], 4: [], 5: [86, 90, 91, 92, 93, 94, 81, 82, 85], 6: [1, 4, 5, 16, 19, 25, 26, 27, 28, 29, 31, 32, 36, 40, 41, 44, 45], 7: [], 8: [87, 88, 89, 80, 95, 83, 84], 9: [3, 6, 7, 8, 9, 11, 12, 13, 20, 23, 30, 33, 35, 37, 39, 42, 43]}

Arbeiterrouten (y):
{0: [3, 49, 93, 82, 84, 21, 29, 32, 37, 39, 41], 1: [86, 5, 94, 57, 59, 20, 69, 38, 42, 44], 2: [8, 10, 62, 64, 23, 70, 28, 74, 34, 36], 3: [89, 6, 9, 55, 85, 15, 67, 77, 79], 4: [88, 81, 16, 26, 40], 5: [2, 90, 50, 52, 56, 58, 22, 71, 73, 31, 43], 6: [0, 46, 4, 92, 80, 54, 12, 14, 60, 18, 72, 30], 7: [47, 91, 95, 17, 63, 65, 25, 75, 35], 8: [87, 48, 7, 53, 11, 19, 24, 27, 76, 78, 45], 9: [1, 51, 83, 13, 61, 66, 68, 33]}

Anbaugeräte-Routen (z):
{0: [7], 1: [6, 8], 2: [46], 3: [], 4: [], 5: [], 6: [], 7: [], 8: [38]}
